# Experiment: Binary and Multi-class Sentiment Framing

Objective: train a binary sentiment classifier and derive a pseudo 3-class setup
(`negative`, `neutral`, `positive`) from calibrated confidence thresholds.


## 1. Setup


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
np.random.seed(42)
random.seed(42)


## 2. Configuration


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.json"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "test.json"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUBMISSION_DIR = PROJECT_ROOT / "data" / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"

for d in [PROCESSED_DIR, SUBMISSION_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RANDOM_SEED = 42
PROJECT_ROOT


## 3. Data Loading


In [ ]:
from sentiment_project.core import LABEL_COL, REVIEW_COL, load_test_dataframe, load_train_dataframe

train_df = load_train_dataframe(TRAIN_PATH)
test_df = load_test_dataframe(TEST_PATH)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


## 4. Data Validation


In [ ]:
assert train_df[LABEL_COL].isin([0, 1]).all()
assert train_df[REVIEW_COL].str.len().gt(0).all()

train_df[LABEL_COL].value_counts(normalize=True).rename("ratio")


## 5. Exploratory Analysis


In [ ]:
train_df["word_len"] = train_df[REVIEW_COL].str.split().str.len()
sns.histplot(data=train_df, x="word_len", hue=LABEL_COL, bins=50, element="step")
plt.title("Word-length by binary sentiment")
plt.show()


## 6. Feature Engineering


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words="english",
    ngram_range=(1, 2),
    max_features=80_000,
    min_df=2,
    sublinear_tf=True,
)


## 7. Modeling


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

pipe = Pipeline(
    [
        ("tfidf", vectorizer),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
    ]
)

x_train, x_val, y_train, y_val = train_test_split(
    train_df[REVIEW_COL], train_df[LABEL_COL], test_size=0.2, stratify=train_df[LABEL_COL], random_state=RANDOM_SEED
)
pipe.fit(x_train, y_train)


## 8. Evaluation


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

val_pred = pipe.predict(x_val)
val_prob = pipe.predict_proba(x_val)[:, 1]

binary_metrics = {
    "accuracy": float(accuracy_score(y_val, val_pred)),
    "macro_f1": float(f1_score(y_val, val_pred, average="macro")),
}
binary_metrics


In [ ]:
# Pseudo 3-class conversion using confidence thresholds.
def to_three_class(prob: float, low: float = 0.4, high: float = 0.6) -> int:
    if prob <= low:
        return 0   # negative
    if prob >= high:
        return 2   # positive
    return 1       # neutral

val_three = pd.Series([to_three_class(p) for p in val_prob])
val_three.value_counts().sort_index().rename(index={0: "neg", 1: "neu", 2: "pos"})


## 9. Inference / Submission


In [ ]:
test_pred = pipe.predict(test_df[REVIEW_COL])
submission = pd.DataFrame({"id": range(len(test_pred)), "sentiments": test_pred.astype(int)})
out = SUBMISSION_DIR / "submission_binary_logreg.csv"
submission.to_csv(out, index=False)
print(out.relative_to(PROJECT_ROOT))
submission.head()


## 10. Conclusions / Next Steps

- Binary sentiment remains the official target for this dataset.
- A pseudo multi-class (`neg/neu/pos`) framing can be created from confidence thresholds when neutral labels are absent.
- This setup is useful for downstream triage and uncertain-case handling.
